# Experiment 12 — Confusion-Aware CORN Loss — Colab Run

Same `mit_b1 + CORN` architecture, data root, image size, augmentation protocol, and 3-fold split (`seed=42`) as Chapter 7's baseline — **no architecture, dataset, or encoder change anywhere**. Only the training loss changes between variants (`scripts/losses.py`):

- **baseline** — exact reproduction of the original `CornLoss` (sanity check + the apples-to-apples comparison target for every other variant).
- **Exp-12A** — `L = L_CORN + lam * L_dist` (ordinal-distance regularization, no confusion statistics — the control).
- **Exp-12B** — `L = L_CORN + lam * L_conf` (confusion-aware, using a frozen per-fold cost matrix built from the *validation* split of the existing baseline checkpoints — never the outer test fold). **This is the experiment's actual research hypothesis.**
- **Exp-12C** — `L = L_CORN + beta * L_3` (extra weight on just the Stage-3/4 CORN threshold — the simple alternative).

Protocol, exactly as planned: run Exp-12A and Exp-12B's small hyperparameter grids on **fold 0's validation set only** first. If 12B doesn't look like it's earning its extra complexity over 12A there, that's already informative — but every selected configuration still gets evaluated on all 3 folds' held-out test sets for the final paired comparison, since a single fold's validation ranking isn't the thesis's reported result. Exp-12C is evaluated the same way, independently.

**No test-fold data is used for hyperparameter selection anywhere in this notebook.**

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

No dataset zip to upload for this experiment — it trains on the same `datasets/wound-stage-classification/` RGB images as Chapter 7's baseline, which the Colab runtime needs available locally (see the `ablation` notebooks for how that gets there; if it's already extracted for another experiment's notebook in this session, this one reuses it).

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

In [ ]:
import glob
import os

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-stage-classification.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-stage-classification"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not os.path.isdir(DATASET_DIR):
    if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}
    candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
    assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
    source = candidates[0]
    os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
    if os.path.islink(DATASET_DIR):
        os.remove(DATASET_DIR)
    os.symlink(source, DATASET_DIR)
print("dataset at:", DATASET_DIR, "exists:", os.path.isdir(DATASET_DIR))
!ls {DATASET_DIR} | head

## 3. Install dependencies

In [ ]:
import os
!pip install -q -r {REPO_DIR}/task_3/requirements.txt
os.chdir(f"{REPO_DIR}/more_experiments/exp/exp12_confusion_aware_loss/scripts")
print(os.getcwd())

## 4. Sanity-check the losses, then (re)build the frozen cost matrices

`test_losses.py` must show ALL CHECKS PASSED before anything below is trusted. `build_cost_matrices.py` only needs the 3 existing baseline checkpoints (`task_3/results/task_3_outputs/checkpoints/mit_b1__corn_fold{0,1,2}.pt`, already in the repo) and each fold's *validation* split — no training, a few seconds on CPU. The results are already committed under `results/cost_matrices/`; this just re-verifies them in this runtime (identical numbers expected).

In [ ]:
!python test_losses.py

In [ ]:
!python build_cost_matrices.py

## 5. Redirect checkpoints/results to Drive

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/exp12_confusion_aware_loss_outputs"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
with open("../configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

# train.py writes results/results.csv, results/selection_results.csv, and
# results/predictions/ relative to EXP_DIR (not configurable via base.yaml) --
# symlink the whole exp's results/ dir to Drive instead, so every one of
# those paths lands there without editing train.py.
import os
os.makedirs(f"{OUTPUT_DIR}/exp_results", exist_ok=True)
local_results = "../results"
if os.path.islink(local_results):
    os.remove(local_results)
elif os.path.isdir(local_results):
    !cp -r {local_results}/* {OUTPUT_DIR}/exp_results/ 2>/dev/null || true
    !rm -rf {local_results}
os.symlink(f"{OUTPUT_DIR}/exp_results", local_results)
print("results now symlinked to:", f"{OUTPUT_DIR}/exp_results")
!ls ../results

## 6. Hyperparameter selection — fold 0, validation only

`--mode select` never touches the outer test split. This runs the brief's Section 4 grids: Exp-12A (3 configs), Exp-12B (6 configs), Exp-12C (2 configs) — 11 short training runs on fold 0.

In [ ]:
import subprocess

def run(cmd):
    print("+", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {' '.join(cmd)}")

FOLD = 0

for lam in [0.05, 0.10, 0.20]:
    run(["python", "-u", "train.py", "--variant", "A", "--fold", str(FOLD), "--mode", "select",
         "--lam", str(lam), "--workers", "4"])

for lam in [0.05, 0.10, 0.20]:
    for alpha in [1, 2]:
        run(["python", "-u", "train.py", "--variant", "B", "--fold", str(FOLD), "--mode", "select",
             "--lam", str(lam), "--alpha", str(alpha), "--workers", "4"])

for beta in [0.5, 1.0]:
    run(["python", "-u", "train.py", "--variant", "C", "--fold", str(FOLD), "--mode", "select",
         "--beta", str(beta), "--workers", "4"])

# the baseline's own fold-0 validation score, for reference in the table below
run(["python", "-u", "train.py", "--variant", "baseline", "--fold", str(FOLD), "--mode", "select",
     "--workers", "4"])

### Read the selection table and choose one configuration per variant

Selection metric is validation macro-F1, per the brief. Look at this table before continuing — the next cell only prints the automatically-best row per variant; it does **not** silently decide anything for you. If Exp-12B's best row doesn't clearly beat Exp-12A's best row (within roughly the baseline's own fold-to-fold noise), say so plainly in the final write-up: that is itself the thesis-relevant finding (confusion weighting earns nothing over plain ordinal distance), not a failure to report.

In [ ]:
import pandas as pd

sel = pd.read_csv("../results/selection_results.csv")
display(sel.sort_values("val_macro_f1", ascending=False))

print("\nBest row per variant (by val_macro_f1):")
best = sel.loc[sel.groupby("variant")["val_macro_f1"].idxmax()]
display(best[["variant", "lam", "alpha", "beta", "val_accuracy", "val_macro_f1", "val_mae", "val_qwk"]])

## 7. Final evaluation — fill in the SELECTED hyperparameters, then run all 3 folds

Edit `SELECTED` below from the table printed above before running this cell. `--mode final` trains the same way (train+val, early-stopped on val) but then evaluates ONCE on the outer test split, saves the checkpoint, and writes every test-set prediction to `results/predictions/`. The baseline is re-trained here too (not reused from `task_3`'s own checkpoint) so the paired fixed/regressed comparison in the next section is an exact apples-to-apples comparison from this same codebase.

In [ ]:
# EDIT THESE from the Section 6 table before running:
SELECTED = {
    "A": {"lam": 0.10},
    "B": {"lam": 0.10, "alpha": 1},
    "C": {"beta": 0.5},
}

for fold in [0, 1, 2]:
    run(["python", "-u", "train.py", "--variant", "baseline", "--fold", str(fold), "--mode", "final",
         "--workers", "4"])
    run(["python", "-u", "train.py", "--variant", "A", "--fold", str(fold), "--mode", "final",
         "--lam", str(SELECTED["A"]["lam"]), "--workers", "4"])
    run(["python", "-u", "train.py", "--variant", "B", "--fold", str(fold), "--mode", "final",
         "--lam", str(SELECTED["B"]["lam"]), "--alpha", str(SELECTED["B"]["alpha"]), "--workers", "4"])
    run(["python", "-u", "train.py", "--variant", "C", "--fold", str(fold), "--mode", "final",
         "--beta", str(SELECTED["C"]["beta"]), "--workers", "4"])

## 8. Aggregate: confusion matrices, stage-pair errors, paired comparison vs. baseline

In [ ]:
CONFIG_IDS = [
    "mit_b1__corn__exp12_baseline",
    f"mit_b1__corn__exp12A_lam{SELECTED['A']['lam']:g}",
    f"mit_b1__corn__exp12B_lam{SELECTED['B']['lam']:g}_alpha{SELECTED['B']['alpha']:g}",
    f"mit_b1__corn__exp12C_beta{SELECTED['C']['beta']:g}",
]

run(["python", "-u", "evaluate.py", "--config-id", CONFIG_IDS[0], "--baseline-predictions", "none"])
for cid in CONFIG_IDS[1:]:
    run(["python", "-u", "evaluate.py", "--config-id", cid])

run(["python", "-u", "generate_report.py", "--config-ids"] + CONFIG_IDS)

In [ ]:
print(open("../results/report.md").read())

## 9. Reference — RGB baseline (Chapter 7 / task_3's original ablation run)

| Metric | Mean ± Std |
|---|---|
| Accuracy | 0.8168 ± 0.0150 |
| Macro-F1 | 0.8182 ± 0.0138 |
| MAE | 0.1896 ± 0.0204 |
| QWK | 0.9138 ± 0.0164 |
| Total errors | 200 / 1091 |
| Stage 3→4 errors | 65 (32.5% of all errors) |
| Stage 4→3 errors | 37 (18.5% of all errors) |

(`task_3/results/task_3_outputs/results/ablation_summary.csv`, `mit_b1__corn` row; stage-pair counts from `more_experiments/reports/figures/mit_b1_corn_test_explanations/all_predictions.csv`.) `exp12_baseline`'s own numbers in Section 8 above are this script's independent reproduction under the exact same loss — compare the two as a sanity check before trusting the A/B/C comparisons.